# FIN exact-algebra campaign: P486 / P485 / P487
Portable executable notebook for Google Colab and Kaggle. The default run does **not** start P475. Every completed or interrupted stage creates a checkpoint ZIP. P485/P487 contain the exact coefficient repair `sqrt(2) = 2 - 4*alpha**2` for `alpha = sin(pi/8)`.

Scientific boundary: a software completion is not automatically a theorem. P485 passes only if all five remainders are exactly zero and P486 premises pass. A P487 polynomial is not yet a minimal polynomial.

In [ ]:
from pathlib import Path
import os, sys, shutil, zipfile, hashlib, json, subprocess
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = Path('/kaggle/working').exists()
print({'python': sys.version, 'colab': IN_COLAB, 'kaggle': IN_KAGGLE})
import numpy, sympy
print('numpy', numpy.__version__, 'sympy', sympy.__version__)

## Persistence choice
For a long Colab run, set `USE_GOOGLE_DRIVE=True`. Kaggle automatically uses `/kaggle/working`; download its checkpoint before the session ends.

In [ ]:
USE_GOOGLE_DRIVE = False  # change to True in Colab for persistent files
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/FIN_exact_algebra_campaign')
elif IN_KAGGLE:
    ROOT = Path('/kaggle/working/FIN_exact_algebra_campaign')
else:
    ROOT = Path.cwd() / 'FIN_exact_algebra_campaign'
ROOT.mkdir(parents=True, exist_ok=True)
print('Persistent work directory:', ROOT)

## Locate or upload the source bundle
In Colab, the cell opens an upload dialog if the ZIP is not already present. In Kaggle, add the ZIP as a notebook input/dataset or upload it to the working directory.

In [ ]:
BUNDLE_NAME = 'FIN_Exact_Algebra_Colab_Kaggle_Bundle.zip'
candidates = [Path.cwd()/BUNDLE_NAME, Path('/content')/BUNDLE_NAME, Path('/kaggle/working')/BUNDLE_NAME]
if IN_KAGGLE:
    candidates += list(Path('/kaggle/input').glob('**/'+BUNDLE_NAME))
bundle = next((p for p in candidates if p.exists()), None)
if bundle is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    if BUNDLE_NAME not in uploaded: raise FileNotFoundError(BUNDLE_NAME)
    bundle = Path('/content')/BUNDLE_NAME
if bundle is None: raise FileNotFoundError('Place '+BUNDLE_NAME+' beside the notebook or in Kaggle input.')
print('Bundle:', bundle, 'SHA256:', hashlib.sha256(bundle.read_bytes()).hexdigest())
with zipfile.ZipFile(bundle) as zf:
    zf.extractall(ROOT)
print('Extracted', len(list(ROOT.iterdir())), 'entries')

## Mandatory preflight
This checks syntax, the exact coefficient identity, source hashes, and imports. It does not start a Gröbner basis.

In [ ]:
os.chdir(ROOT)
required = ['fin_phase_exact_algebra.py','fin_program_486.py','fin_program_485_unlimited.py','fin_program_487_unlimited.py','fin_program_475_unlimited.py','fin_remote_campaign_runner.py','FIN_Program_480_Standalone_Certificate.json']
missing = [name for name in required if not Path(name).exists()]
assert not missing, missing
subprocess.run([sys.executable, '-m', 'py_compile', *required[:-1]], check=True)
import sympy as sp
alpha = sp.sin(sp.pi/8)
assert sp.simplify(sp.sqrt(2) - (2-4*alpha**2)) == 0
manifest = json.loads(Path('FIN_Remote_Bundle_Manifest.json').read_text())
for name, expected in manifest['sha256'].items():
    actual = hashlib.sha256(Path(name).read_bytes()).hexdigest()
    assert actual == expected, (name, expected, actual)
print('PREFLIGHT PASS: syntax, hashes, and coefficient closure verified.')

## Choose the campaign
Recommended: keep `P486,P485,P487`. Do not add P475 unless you intentionally want the old, much heavier fourteen-variable cross-check.

In [ ]:
PROGRAMS = 'P486,P485,P487'  # optional explicit extension: ...,P475
print('Selected:', PROGRAMS)

## Run
Interrupting this cell forwards SIGINT to the active algebra process and creates a stopped checkpoint. Colab may still terminate idle/disconnected runtimes; Google Drive persistence is recommended.

In [ ]:
command = [sys.executable, '-u', 'fin_remote_campaign_runner.py', '--programs', PROGRAMS, '--root', str(ROOT)]
print('Running:', ' '.join(command))
completed = subprocess.run(command, cwd=ROOT)
print('Runner exit code:', completed.returncode)

## Inspect and download results

In [ ]:
state_path = ROOT/'FIN_Remote_Campaign_State.json'
if state_path.exists(): print(json.dumps(json.loads(state_path.read_text()), indent=2))
archives = sorted(ROOT.glob('FIN_Remote_Checkpoint_*.zip'), key=lambda p:p.stat().st_mtime)
print('Checkpoints:', [(p.name, p.stat().st_size) for p in archives])
if IN_COLAB and archives:
    from google.colab import files
    files.download(str(archives[-1]))
elif archives:
    print('Download manually:', archives[-1])